In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from langchain_core.tools import tool
import requests
import os

EXCHANGE_RATE_KEY = os.getenv('EXCHANGE_RATE_API_KEY')


@tool(
    description="""
    Get the currency conversion rate between a base currency and target currency.
    Use this tool first when a user asks to convert currencies. Pass its
    conversion rate result to the convert tool.
    """
)
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    
    url = f'https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_KEY}/pair/{base_currency}/{target_currency}'
    
    raw_result = requests.get(url)
    data = raw_result.json()
    return data['conversion_rate']

@tool(
    description="""
    Convert a currency amount using a conversion rate obtained from
    get_conversion_factor. Use this tool after getting the conversion rate.
    """
)
def convert(base_currency_value: float, conversion_rate: float) -> float:
    return base_currency_value * conversion_rate

In [7]:
from langchain_groq import ChatGroq

llm = ChatGroq(model='openai/gpt-oss-120b',temperature=0.1)

In [8]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[get_conversion_factor,convert],
)   

query = HumanMessage('What is 20 dollars equivalent to indian rupee?')

initial_state = {"messages": [query]}
result = agent.invoke(initial_state)

In [19]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

for msg in result["messages"]:

    if isinstance(msg, HumanMessage):
        print(f"User: {msg.content}")
        print("=" * 60)

    elif isinstance(msg, AIMessage):

        reasoning = msg.additional_kwargs.get("reasoning_content")

        if reasoning:
            print(f"Thought:\n{reasoning}")

        if msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"Action: {tool_call['name']}")
                print(f"Args: {tool_call['args']}")
                print()

        elif msg.content:
            print(f"Final Answer: {msg.content}")
            print("=" * 60)

    elif isinstance(msg, ToolMessage):
        print(f"Observation ({msg.name}): {msg.content}")
        print("-" * 60)

User: What is 20 dollars equivalent to indian rupee?
Thought:
User asks: "What is 20 dollars equivalent to indian rupee?" Need to get conversion rate USD to INR. Use get_conversion_factor then convert.
Action: get_conversion_factor
Args: {'base_currency': 'USD', 'target_currency': 'INR'}

Observation (get_conversion_factor): 94.5176
------------------------------------------------------------
Thought:
We have conversion rate: 1 USD = 94.5176 INR. Need to convert 20 USD to INR: 20 * 94.5176 = 1890.352. Use convert tool.
Action: convert
Args: {'base_currency_value': 20, 'conversion_rate': 94.5176}

Observation (convert): 1890.352
------------------------------------------------------------
Final Answer: 20 USD is approximately **1,890.35 INR** (based on the current conversion rate of 1 USD ≈ 94.52 INR).
